## Install dependencies
Installs the libraries listed in `requirements.txt` (only needed once).

In [ ]:
%pip install -r requirements.txt

## Download the data
Downloads the FER2013 dataset from Kaggle into `data/fer2013`, only if it is not already there.

In [ ]:
import kagglehub
import os

# Download latest version

if not os.path.exists("data/fer2013/train"):
    kagglehub.dataset_download("msambare/fer2013", output_dir="data/fer2013")

print("Dataset ready")

## LBP feature extraction
- Each 48×48 image is split into **16 blocks** (4×4 grid).
- On each block: **LBP** (8 neighbours, radius 1), then a normalized 256-bin histogram.
- The 16 histograms are concatenated: each image becomes a vector of **4096 values**.

This is applied to every image in `train` and `test`.

In [ ]:

from skimage import io
import matplotlib.pyplot as plt

from skimage.feature import local_binary_pattern
import numpy as np

dossier = "data/fer2013/train"
classes = sorted(
    c for c in os.listdir(dossier)
    if os.path.isdir(os.path.join(dossier, c))
)
print(classes)

# LBD + Histogrammes

N_BLOCS = 16
grille = int(np.sqrt(N_BLOCS))

def caracteristiques_lbp(img, grille=grille, P=8, R=1):
    h, w = img.shape
    taille_h, taille_w = h // grille, w // grille
    histogrammes = []
    for i in range(grille):
        for j in range(grille):
            bloc = img[i*taille_h:(i+1)*taille_h, j*taille_w:(j+1)*taille_w]
            lbp_bloc = local_binary_pattern(bloc, P, R, method="default")
            hist, _ = np.histogram(lbp_bloc, bins=256, range=(0, 256))
            histogrammes.append(hist / hist.sum())
    return np.concatenate(histogrammes)

# Import images of the train repository

def charger_dataset(dossier):
    X, y = [], []
    classes = sorted(c for c in os.listdir(dossier)
                     if os.path.isdir(os.path.join(dossier, c)))
    for label in classes:
        dossier_classe = os.path.join(dossier, label)
        fichiers = [f for f in os.listdir(dossier_classe)
                    if f.lower().endswith((".png", ".jpg", ".jpeg"))]
        for f in fichiers:
            img = io.imread(os.path.join(dossier_classe, f))
            X.append(caracteristiques_lbp(img))
            y.append(label)
        print(f"{label} : {len(fichiers)} images")
    return np.array(X), np.array(y)


X_train, Y_train = charger_dataset("data/fer2013/train")
print("X :", X_train.shape, "| y :", Y_train.shape)
X_test, Y_test = charger_dataset("data/fer2013/test")
print("X :", X_test.shape, "| y :", Y_test.shape)


## Sanity check
Plots the LBP vector of the first training image and its emotion.

In [ ]:
# Test

print(Y_train[0])           
plt.plot(X_train[0])
plt.title(f"Vecteur LBP de la 1re image ({Y_train[0]})")
plt.show()

## KNN classification
Trains a KNN (k = 3) on the training set, predicts the test emotions and computes the accuracy.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, Y_train)

y_pred = knn.predict(X_test)
print("Précision :", accuracy_score(Y_test, y_pred))

## Confusion matrix
The diagonal gives the recognition rate of each emotion.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


cm = confusion_matrix(Y_test, y_pred, labels=classes, normalize="true")
ConfusionMatrixDisplay(cm, display_labels=classes).plot(cmap="Blues", xticks_rotation=45)
plt.title("Matrice de confusion LBP + KNN")
plt.tight_layout()
plt.show()

## Save the model
Saves the trained KNN and the LBP parameters (grid, P, R) into `modele_lbp_knn.joblib`, so the app can reuse them without retraining.

In [ ]:
import joblib

modele = {"knn": knn, "grille": grille, "P": 8, "R": 1}
joblib.dump(modele, "modele_lbp_knn.joblib", compress=3)

print("Modèle sauvegardé")

## Predict the emotion of a new image
Converts the image to grayscale 48×48, extracts its LBP features with the same function as for training, and returns the predicted emotion.

In [ ]:
from skimage.color import rgb2gray
from skimage.transform import resize

def predire_emotion(chemin_image, modele):
    img = io.imread(chemin_image)

    # gray scale
    if img.ndim == 3:
        img = (rgb2gray(img[..., :3]) * 255).astype(np.uint8)

    # 48x48 
    img = resize(img, (48, 48), anti_aliasing=True, preserve_range=True).astype(np.uint8)

    vecteur = caracteristiques_lbp(img, grille=modele["grille"], P=modele["P"], R=modele["R"])
    return modele["knn"].predict([vecteur])[0]


# Test
modele_charge = joblib.load("modele_lbp_knn.joblib")
chemin = "data/fer2013/test/happy/" + os.listdir("data/fer2013/test/happy")[0]
print("Émotion prédite :", predire_emotion(chemin, modele_charge))